In [1]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

# 1. Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32 # Increased for better gradient stability
EPOCHS_STAGE1 = 10
EPOCHS_STAGE2 = 30

# 2. Enhanced Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

# Use actual preprocessing function for validation
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    "dataset/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    "dataset/val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

# 3. Model Architecture
def build_model():
    base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

    # Freeze the base model for stage 1
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    output = Dense(3, activation="softmax")(x)

    return Model(inputs=base_model.input, outputs=output), base_model

model, base_model = build_model()

# 4. Stage 1: Training the Head Only
model.compile(optimizer=Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("Starting Stage 1: Training Top Layers...")
model.fit(train_gen, epochs=EPOCHS_STAGE1, validation_data=val_gen)

# 5. Stage 2: Full Model Fine-Tuning
print("Starting Stage 2: Fine-Tuning Entire Model...")
base_model.trainable = True # Unfreeze all layers

# Use a very small learning rate for fine-tuning
model.compile(optimizer=Adam(learning_rate=1e-5),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
              metrics=['accuracy'])

# Advanced Callbacks
callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=1),
    EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', save_best_only=True)
]

history = model.fit(
    train_gen,
    epochs=EPOCHS_STAGE2,
    validation_data=val_gen,
    callbacks=callbacks
)

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/train'

In [2]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# 1. SETUP PATHS (Adjust these to your local machine)
TRAIN_PATH = "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/train"
VAL_PATH = "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/val"

# 2. ENHANCED AUGMENTATION
# Adding brightness and shear helps the model generalize better to different X-ray exposures
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    TRAIN_PATH, target_size=(224, 224), batch_size=32, class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    VAL_PATH, target_size=(224, 224), batch_size=32, class_mode="categorical"
)

# 3. ARCHITECTURE OPTIMIZATION
def build_model():
    base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

    # Stage 1: Freeze base model
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x) # Stabilizes the activations
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    output = Dense(3, activation="softmax")(x)

    return Model(inputs=base_model.input, outputs=output), base_model

model, base_model = build_model()

# 4. TRAINING STAGE 1: Top Layers (Warm-up)
model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_gen, epochs=10, validation_data=val_gen)

# 5. TRAINING STAGE 2: Full Fine-Tuning with Label Smoothing
# Unfreeze the last 100 layers of DenseNet for specific medical feature extraction
base_model.trainable = True
for layer in base_model.layers[:-100]:
    layer.trainable = False

# Label smoothing prevents overconfidence and helps reach higher accuracy
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=1),
    EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True)
]

model.fit(train_gen, epochs=30, validation_data=val_gen, callbacks=callbacks)

Found 4835 images belonging to 3 classes.
Found 1031 images belonging to 3 classes.
Epoch 1/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.8614 - loss: 0.3952 - val_accuracy: 0.9001 - val_loss: 0.3209
Epoch 2/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 200s 1s/step - accuracy: 0.9026 - loss: 0.2809 - val_accuracy: 0.9059 - val_loss: 0.2830
Epoch 3/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 206s 1s/step - accuracy: 0.9090 - loss: 0.2631 - val_accuracy: 0.9117 - val_loss: 0.2559
Epoch 4/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 215s 1s/step - accuracy: 0.9229 - loss: 0.2284 - val_accuracy: 0.9360 - val_loss: 0.2133
Epoch 5/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 212s 1s/step - accuracy: 0.9253 - loss: 0.2287 - val_accuracy: 0.9059 - val_loss: 0.2588
Epoch 6/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 217s 1s/step - accuracy: 0.9309 - loss: 0.2053 - val_accuracy: 0.9253 - val_loss: 0.2294
Epoch 7/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 235s 2s/step - accuracy: 0.9340 - loss: 0.1914 - val_accuracy: 0.9321 - val_loss: 0.2244
Epoch 8/10
15

In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.densenet import preprocess_input # Change this if using EfficientNet
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Set the path to your test dataset
TEST_PATH = "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/test"

# 2. Define the generator
# IMPORTANT: 'shuffle=False' is required to keep labels in the correct order for the report
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_gen = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

print("Test generator defined successfully.")

Found 1036 images belonging to 3 classes.
Test generator defined successfully.


In [7]:
# 1. Get predictions
print("Predicting on test data...")
preds = model.predict(test_gen)

# 2. Convert predictions to class indices (0, 1, or 2)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes
class_labels = list(test_gen.class_indices.keys())

# 3. Print Accuracy and Classification Report
accuracy = accuracy_score(y_true, y_pred)
print(f"\nOverall Test Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_labels))

# 4. Generate Confusion Matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred)
print(cm)

Predicting on test data...
33/33 ━━━━━━━━━━━━━━━━━━━━ 31s 900ms/step

Overall Test Accuracy: 93.44%

Classification Report:
              precision    recall  f1-score   support

       covid       0.99      0.94      0.96       340
      normal       0.86      0.98      0.92       348
   pneumonia       0.97      0.88      0.92       348

    accuracy                           0.93      1036
   macro avg       0.94      0.93      0.94      1036
weighted avg       0.94      0.93      0.93      1036


Confusion Matrix:
[[320  14   6]
 [  4 341   3]
 [  0  41 307]]


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D, Dense, Dropout, BatchNormalization, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
from tensorflow.keras.regularizers import l2
import numpy as np

# 1. ENHANCED AUGMENTATION (Added contrast and brightness)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2], # Crucial for varying X-ray exposures
    fill_mode="constant",
    cval=0
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    TRAIN_PATH, target_size=(224, 224), batch_size=32, class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    VAL_PATH, target_size=(224, 224), batch_size=32, class_mode="categorical"
)

# 2. ARCHITECTURE WITH DUAL POOLING & L2 REGULARIZATION
def build_model():
    base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False

    x = base_model.output
    # Capture both global patterns and sharp localized features
    gap = GlobalAveragePooling2D()(x)
    gmp = GlobalMaxPooling2D()(x)
    x = Concatenate()([gap, gmp])

    x = BatchNormalization()(x)
    x = Dense(512, activation="relu", kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation="relu", kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.3)(x)

    output = Dense(3, activation="softmax")(x)
    return Model(inputs=base_model.input, outputs=output), base_model

model, base_model = build_model()

# 3. STAGE 1: Warm-up (Increased Learning Rate)
model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_gen, epochs=5, validation_data=val_gen)

# 4. STAGE 2: Deep Fine-Tuning
# Unfreeze from the start of the 5th Dense Block
base_model.trainable = True
set_trainable = False
for layer in base_model.layers:
    if layer.name == 'conv5_block1_0_relu':
        set_trainable = True
    if not set_trainable:
        layer.trainable = False

# Cosine Decay Schedule for smoother convergence
def lr_schedule(epoch):
    initial_lr = 1e-5
    return initial_lr * 0.5 * (1 + np.cos(np.pi * epoch / 30))

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)



callbacks = [
    LearningRateScheduler(lr_schedule),
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
]

# 5. FINAL TRAINING
model.fit(train_gen, epochs=30, validation_data=val_gen, callbacks=callbacks)

Found 4835 images belonging to 3 classes.
Found 1031 images belonging to 3 classes.
Epoch 1/5
 15/152 ━━━━━━━━━━━━━━━━━━━━ 2:10 955ms/step - accuracy: 0.5879 - loss: 2.1018